# 06-06. Carbon Cycle in PSLNMAD  
# PSLNMAD で炭素循環を考える

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

## 今日の目的 / Goals

06-05 では、PSLNMAD の Ideal Age を計算しました。  
In 06-05, we calculated Ideal Age in PSLNMAD.

この Notebook では、PSLNMAD に **炭素循環** を入れます。  
In this notebook, we add the **carbon cycle** to PSLNMAD.

扱う変数は次です。  
We treat:

```text
DIC
PO4
O2
```

特に注目するのは、A と D の違いです。  
We focus especially on the difference between A and D.

> **A と D を分けることで、深層炭素貯蔵を一枚岩ではなく、水塊経路と年齢を持つ構造として考えられる。**  
> **By separating A and D, deep-ocean carbon storage can be viewed not as one uniform reservoir, but as a structured system with pathways and ages.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: なぜ 7-box で炭素循環を見るのか / Why study carbon cycling in 7-box?

PSLNMAD では、深層を

```text
A: 北大西洋起源深層水の経路
D: より古い深層水
```

に分けます。  
In PSLNMAD, the deep ocean is separated into:

```text
A: Atlantic deep pathway
D: older deep water
```

これにより、A と D の DIC, \( \mathrm{PO4} \), O2 の違いを考えられます。  
This lets us examine differences in DIC, \( \mathrm{PO4} \), and O2 between A and D.

## 2. 炭素循環の基本 / Basic carbon-cycle idea

表層では、生物生産によって DIC と \( \mathrm{PO4} \) が取り除かれます。  
At the surface, biological production removes DIC and \( \mathrm{PO4} \).

内部では、有機物が再無機化され、DIC と \( \mathrm{PO4} \) が増え、O2 が減ります。  
In the interior, organic matter is remineralized, increasing DIC and \( \mathrm{PO4} \), while decreasing O2.

```text
surface:
    DIC decreases
    PO4 decreases

interior:
    DIC increases
    PO4 increases
    O2 decreases
```

## 3. PSLNMAD の準備 / Prepare PSLNMAD

In [ ]:
boxes = ["P", "S", "L", "N", "M", "A", "D"]
surface_boxes = ["P", "S", "L", "N"]
interior_boxes = ["M", "A", "D"]

VOCN_TOTAL = 1.292e18
VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
    "A": 0.12 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())

volumes = np.array([VOLUME[b] for b in boxes])
idx = {b: i for i, b in enumerate(boxes)}

Q_DEFAULT = 2.0e7
DT = 8.64e4
SEC_PER_YEAR = 365 * 86400
DT_YEAR = DT / SEC_PER_YEAR

pathway = [("N", "A"), ("A", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]
exchanges_default = [("S", "M", 0.3e7), ("L", "M", 0.2e7)]

pd.DataFrame({
    "Box": boxes,
    "Layer": ["surface", "surface", "surface", "surface", "mid-depth", "Atlantic deep pathway", "deep"],
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in boxes],
})

## 4. 輸送行列 / Transport matrix

In [ ]:
def build_flux_matrix(pathway, Q, boxes):
    F = np.zeros((len(boxes), len(boxes)))
    local_idx = {b: i for i, b in enumerate(boxes)}
    for source, target in pathway:
        i = local_idx[target]
        j = local_idx[source]
        F[i, j] += Q
        F[j, j] -= Q
    return F

def build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges=None):
    F = build_flux_matrix(pathway, Q, boxes)
    local_idx = {b: i for i, b in enumerate(boxes)}
    if exchanges is None:
        exchanges = []
    for a, b, q in exchanges:
        ia = local_idx[a]
        ib = local_idx[b]
        F[ib, ia] += q
        F[ia, ia] -= q
        F[ia, ib] += q
        F[ib, ib] -= q
    return F

F = build_flux_matrix_with_exchange(pathway, Q_DEFAULT, boxes, exchanges_default)
pd.DataFrame(F, index=boxes, columns=boxes)

## 5. 初期値 / Initial state

ここでは教育用の単純な初期値を使います。  
Here we use simple teaching values.

In [ ]:
def initial_state():
    state = {}
    for b in boxes:
        state[f"DIC_{b}"] = 2200.0
        state[f"PO4_{b}"] = 2.0
        state[f"O2_{b}"] = 220.0
    return state

state0 = initial_state()

pd.DataFrame({
    "Box": boxes,
    "DIC": [state0[f"DIC_{b}"] for b in boxes],
    "PO4": [state0[f"PO4_{b}"] for b in boxes],
    "O2": [state0[f"O2_{b}"] for b in boxes],
})

## 6. Redfield 比 / Redfield ratio

ここでは、

```text
C:P = 106
O2:P = 172
```

を使います。  
We use:

```text
C:P = 106
O2:P = 172
```

\( \mathrm{PO4} \) が 1 増えると、DIC は 106 増え、O2 は 172 減ります。  
When \( \mathrm{PO4} \) increases by 1, DIC increases by 106 and O2 decreases by 172.

In [ ]:
RCP = 106.0
RO2P = 172.0

delta_po4 = 0.1
print("If PO4 increases by", delta_po4)
print("DIC increases by", RCP * delta_po4)
print("O2 decreases by", RO2P * delta_po4)

### Mini exercise 1 / ミニ演習 1

再無機化によって、DIC, \( \mathrm{PO4} \), O2 はそれぞれどう変化しますか。  
How do DIC, \( \mathrm{PO4} \), and O2 change during remineralization?

## 7. 生物ポンプの source/sink / Biological source/sink terms

表層から有機物が輸出され、M, A, D で再無機化されるとします。  
Organic matter is exported from the surface and remineralized in M, A, and D.

In [ ]:
def biological_terms(export_scale=1.0):
    po4_tendency = {b: 0.0 for b in boxes}

    export = {
        "P": 0.15 * export_scale,
        "S": 0.08 * export_scale,
        "L": 0.25 * export_scale,
        "N": 0.10 * export_scale,
    }

    for b, e in export.items():
        po4_tendency[b] -= e

    total_export = sum(export.values())
    po4_tendency["M"] += 0.35 * total_export
    po4_tendency["A"] += 0.25 * total_export
    po4_tendency["D"] += 0.40 * total_export

    dic_tendency = {b: RCP * po4_tendency[b] for b in boxes}
    o2_tendency = {b: -RO2P * po4_tendency[b] for b in boxes}

    return dic_tendency, po4_tendency, o2_tendency

dic_bio, po4_bio, o2_bio = biological_terms(export_scale=1.0)

pd.DataFrame({
    "Box": boxes,
    "DIC bio tendency": [dic_bio[b] for b in boxes],
    "PO4 bio tendency": [po4_bio[b] for b in boxes],
    "O2 bio tendency": [o2_bio[b] for b in boxes],
})

## 8. 1 step 更新関数 / One-step update function

In [ ]:
def get_array(state, name):
    return np.array([state[f"{name}_{b}"] for b in boxes], dtype=float)

def put_array(state, name, arr):
    for i, b in enumerate(boxes):
        state[f"{name}_{b}"] = arr[i]

def one_step_transport(c, F):
    return c + (F @ c) * DT / volumes

def one_step_carbon(state, F, export_scale=1.0):
    new = dict(state)
    dic_bio, po4_bio, o2_bio = biological_terms(export_scale=export_scale)

    for name, bio_terms in [("DIC", dic_bio), ("PO4", po4_bio), ("O2", o2_bio)]:
        arr = get_array(state, name)
        arr = one_step_transport(arr, F)
        for i, b in enumerate(boxes):
            arr[i] += bio_terms[b] * DT_YEAR
        put_array(new, name, arr)

    return new

state1 = one_step_carbon(state0, F, export_scale=1.0)

pd.DataFrame({
    "Box": boxes,
    "DIC before": [state0[f"DIC_{b}"] for b in boxes],
    "DIC after one day": [state1[f"DIC_{b}"] for b in boxes],
    "O2 before": [state0[f"O2_{b}"] for b in boxes],
    "O2 after one day": [state1[f"O2_{b}"] for b in boxes],
})

## 9. 標準実験 / Standard experiment

In [ ]:
def run_carbon(years=3000, Q=Q_DEFAULT, exchanges=exchanges_default, export_scale=1.0):
    F_local = build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges)
    state = initial_state()
    hist = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365}
            for b in boxes:
                row[f"DIC_{b}"] = state[f"DIC_{b}"]
                row[f"PO4_{b}"] = state[f"PO4_{b}"]
                row[f"O2_{b}"] = state[f"O2_{b}"]
            hist.append(row)
        state = one_step_carbon(state, F_local, export_scale=export_scale)

    return pd.DataFrame(hist)

hist_std = run_carbon(years=3000)

plt.figure()
for b in boxes:
    plt.plot(hist_std["year"], hist_std[f"DIC_{b}"], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("DIC")
plt.title("DIC in PSLNMAD")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
for b in boxes:
    plt.plot(hist_std["year"], hist_std[f"O2_{b}"], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("O2")
plt.title("O2 in PSLNMAD")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
for b in boxes:
    plt.plot(hist_std["year"], hist_std[f"PO4_{b}"], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("PO4")
plt.title("PO4 in PSLNMAD")
plt.legend()
plt.grid(True)
plt.show()

## 10. A と D を比較する / Compare A and D

A は比較的新しい北大西洋起源深層水の経路、D はより古い深層水です。  
A is the relatively young Atlantic deep pathway, while D is older deep water.

一般には D の方が DIC と \( \mathrm{PO4} \) が高く、O2 が低くなりやすいと考えられます。  
In general, D tends to have higher DIC and \( \mathrm{PO4} \), and lower O2.

In [ ]:
plt.figure()
plt.plot(hist_std["year"], hist_std["DIC_A"], label="DIC A")
plt.plot(hist_std["year"], hist_std["DIC_D"], label="DIC D")
plt.xlabel("Time [yr]")
plt.ylabel("DIC")
plt.title("DIC: A vs D")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(hist_std["year"], hist_std["O2_A"], label="O2 A")
plt.plot(hist_std["year"], hist_std["O2_D"], label="O2 D")
plt.xlabel("Time [yr]")
plt.ylabel("O2")
plt.title("O2: A vs D")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Box": ["A", "D"],
    "Final DIC": [hist_std["DIC_A"].iloc[-1], hist_std["DIC_D"].iloc[-1]],
    "Final PO4": [hist_std["PO4_A"].iloc[-1], hist_std["PO4_D"].iloc[-1]],
    "Final O2": [hist_std["O2_A"].iloc[-1], hist_std["O2_D"].iloc[-1]],
})

### Mini exercise 2 / ミニ演習 2

A と D では、DIC と O2 はどちらが高くなりましたか。  
Between A and D, which has higher DIC and which has higher O2?

## 11. 生物ポンプ感度 / Biological pump sensitivity

In [ ]:
export_list = [0.0, 0.5, 1.0, 2.0]
summary_export = []

plt.figure()
for es in export_list:
    h = run_carbon(years=3000, export_scale=es)
    plt.plot(h["year"], h["DIC_D"], label=f"export={es}")
    summary_export.append({
        "export_scale": es,
        "Final DIC_A": h["DIC_A"].iloc[-1],
        "Final DIC_D": h["DIC_D"].iloc[-1],
        "Final O2_A": h["O2_A"].iloc[-1],
        "Final O2_D": h["O2_D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("DIC in D")
plt.title("Biological pump sensitivity")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_export)

## 12. ベンチレーション強度 Q の感度 / Sensitivity to ventilation strength Q

In [ ]:
Q_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary_Q = []

plt.figure()
for q in Q_list:
    h = run_carbon(years=3000, Q=q, export_scale=1.0)
    plt.plot(h["year"], h["O2_D"], label=f"Q={q:.1e}")
    summary_Q.append({
        "Q": q,
        "Final DIC_A": h["DIC_A"].iloc[-1],
        "Final DIC_D": h["DIC_D"].iloc[-1],
        "Final O2_A": h["O2_A"].iloc[-1],
        "Final O2_D": h["O2_D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("O2 in D")
plt.title("Ventilation sensitivity")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_Q)

## 13. DIC と O2 の関係 / Relationship between DIC and O2

In [ ]:
last = hist_std.iloc[-1]

df_final = pd.DataFrame({
    "Box": boxes,
    "DIC": [last[f"DIC_{b}"] for b in boxes],
    "O2": [last[f"O2_{b}"] for b in boxes],
    "PO4": [last[f"PO4_{b}"] for b in boxes],
})

plt.figure()
plt.scatter(df_final["DIC"], df_final["O2"])
for _, row in df_final.iterrows():
    plt.text(row["DIC"], row["O2"], row["Box"])
plt.xlabel("DIC")
plt.ylabel("O2")
plt.title("DIC-O2 relationship")
plt.grid(True)
plt.show()

df_final

## 14. 06-06 のまとめ / Summary of 06-06

この Notebook で学んだことは次です。  
What we learned:

1. PSLNMAD に DIC, \( \mathrm{PO4} \), O2 を導入した。  
   We introduced DIC, \( \mathrm{PO4} \), and O2 into PSLNMAD.

2. 表層では生物ポンプにより DIC と \( \mathrm{PO4} \) が減る。  
   At the surface, the biological pump reduces DIC and \( \mathrm{PO4} \).

3. 内部では再無機化により DIC と \( \mathrm{PO4} \) が増え、O2 が減る。  
   In the interior, remineralization increases DIC and \( \mathrm{PO4} \), and decreases O2.

4. A と D を分けることで、北大西洋起源深層水と古い深層水の炭素循環の違いを見られる。  
   Separating A and D allows us to examine carbon-cycle differences between North Atlantic-origin deep water and older deep water.

## Key message

> **PSLNMAD では、深層炭素貯蔵を A と D に分けて考えることで、ベンチレーション・再無機化・炭素蓄積をより明確に理解できる。**  
> **In PSLNMAD, separating deep carbon storage into A and D clarifies ventilation, remineralization, and carbon accumulation.**

## 15. 課題 / Exercises

### 課題 1 / Exercise 1

生物ポンプによって、表層と内部の DIC, \( \mathrm{PO4} \), O2 はどう変化するか。  
How does the biological pump change DIC, \( \mathrm{PO4} \), and O2 in the surface and interior?

### 課題 2 / Exercise 2

A と D を分けることで、炭素循環について何が見やすくなるか。  
What becomes easier to examine in carbon cycling by separating A and D?

### 課題 3 / Exercise 3

再無機化により、DIC と O2 はなぜ逆向きに変化するか。  
Why do DIC and O2 change in opposite directions during remineralization?

### 課題 4 / Exercise 4

生物ポンプを強くすると、D の DIC と O2 はどう変わるか。  
How do DIC and O2 in D change when the biological pump is strengthened?

### 課題 5 / Exercise 5

ベンチレーションが強くなると、D の DIC と O2 はどう変わると考えられるか。  
How do DIC and O2 in D change when ventilation becomes stronger?

## 16. 次へ / Next step

次の Notebook では、PSLNMAD に炭素同位体を導入します。  
In the next notebook, we introduce carbon isotopes into PSLNMAD.

```text
06-07 δ13C and Δ14C in PSLNMAD
```